### Some more preprocessing

In [0]:
%pip install --upgrade typing_extensions transformers torch
%pip install sentence-transformers

Python interpreter will be restarted.
  Using cached typing_extensions-4.14.0-py3-none-any.whl (43 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.1.1
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5340536d-02d2-4142-814b-1f85a489c6e9
    Can't uninstall 'typing-extensions'. No files were found to uninstall.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.


In [0]:
from pyspark.sql.functions import col, count, collect_list, struct
from transformers import pipeline
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, FloatType, IntegerType
from sentence_transformers import SentenceTransformer, util
import numpy as np
import plotly.graph_objects as go

In [0]:
%sql
SELECT COUNT(*) FROM amd_with_sentiment

count(1)
70


In [0]:
df = spark.sql("SELECT * FROM amd_with_sentiment")


In [0]:
df.columns

Out[4]: ['Date',
 'headline',
 'summary',
 'Close_NVDA',
 'High_NVDA',
 'Low_NVDA',
 'Open_NVDA',
 'Volume_NVDA',
 'Daily_Gap',
 'sentiment',
 'sentiment_score']

In [0]:
display(df)

Date,headline,summary,Close_NVDA,High_NVDA,Low_NVDA,Open_NVDA,Volume_NVDA,Daily_Gap,sentiment,sentiment_score
2024-06-20,"Housing trends, HPE CEO on Nvidia partnership: Market Domination","There's never enough time in the day to trade, as Market Domination Hosts Julie Hyman and Josh Lipton walk investors through the final trading hour of Thursday, June 20. They cover the top trending stocks and market movements ahead of the closing bell. Hewlett Packard Enterprise (HPE) CEO Antonio Neri discusses HPE's new partnership with Nvidia (NVDA) on its line of ""Nvidia AI Computing by HPE"" product offerings. National Association of Home Builders (NAHB) CEO Jim Tobin stops into the studio to tackle some of the biggest challenges the US housing market is currently facing. Yahoo Finance's top trending stock tickers this hour include Gilead Sciences (GILD), commercial-grade EV maker Nikola (NKLA), and Advanced Micro Devices (AMD). This post was written by Luke Carberry Mogan.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Neutral,0.9999926
2024-06-20,AMD named top pick at Piper Sandler,"Shares of Advanced Micro Devices (AMD) popped on Thursday. Piper Sandler analyst Harsh Kumar named the stock as a top pick in the large-cap space. He cites a number of factors, including the competitive positioning of its MI products, and says that he sees AMD ""as having bright prospects moving into the back half of the year."" Kumar has an Overweight rating and $175 price target on the stock. Yahoo Finance's Josh Lipton and Julie Hyman discuss the call in the video above. For more expert insight and the latest market action, click here to watch this full episode of Market Domination. This post was written by Stephanie Mikulich.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.99999976
2024-06-20,AMD Shares Now Top Piper Pick Because of AI Server Chips,Shares of Advanced Micro Devices traded higher Thursday after a Pipe Sandler analyst issued a bullish note on the artificial intelligence business of the chip maker.,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.9999987
2024-06-20,Why Are AMD (AMD) Shares Soaring Today,"Shares of computer processor maker AMD (NASDAQ:AMD) jumped 7.5% in the morning session after Piper Sandler analyst Harsh Kumar named the company a ""Top Pick"" and maintained an Overweight (Buy) rating on the stock.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,1
2024-06-20,AMD Stock Is Set Up For Another Spike,AMD's upcoming quarterly earnings report on August 6th could serve as a catalyst for a potential spike in stock price. Find out if AMD stock is a buy.,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.9999999
2024-06-20,Piper Sandler names AMD as top large-cap pick into 2H24,"Piper Sandler analysts named AMD (NASDAQ: NASDAQ:AMD) their top large-cap pick for the second half of 2024, citing positive feedback from their discussions with the chipmaker's management in Europe last week.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.9999999
2024-06-20,"AMD Hack Won’t Have a Material Impact on Business, Company Says","(Bloomberg) -- Advanced Micro Devices Inc. found hackers made off with limited information during a recent cyberattack, saying the infiltration shouldn’t have significant impact on its operations.Most Read from BloombergCar Dealerships Across US Halt Services After CyberattackPutin’s Hybrid War Opens a Second Front on NATO’s Eastern BorderHedge Fund Talent Schools Are Looking for the Perfect TraderCar Dealers Across US Are Crippled by a Second CyberattackWhat to Know About the Deadly Flesh-Eatin",130.74778747558594,140.72532511567726

In [0]:
df.groupBy("sentiment").count().show()

+---------+-----+
|sentiment|count|
+---------+-----+
|  Neutral|   24|
| Positive|   40|
| Negative|    6|
+---------+-----+



Remove the neutral news tags as that most likely wont cause changes in stock prices

In [0]:
df = df.filter(df["sentiment"] != "Neutral")
df.count()

Out[7]: 46

See how many summary's sentiments weren't that confidant

In [0]:
df.filter(col("sentiment_score") < 0.80).count()

Out[8]: 2

Show how many news articles we have per date

In [0]:
duplicates = df.groupBy("Date").agg(count("*").alias("count")).filter("count > 1")
display(duplicates)

Date,count
2024-07-11,2
2024-06-20,5
2024-07-30,16
2025-01-07,4
2024-08-01,4
2024-08-05,4
2024-09-03,3
2024-08-07,5


In [0]:
df.columns

Out[10]: ['Date',
 'headline',
 'summary',
 'Close_NVDA',
 'High_NVDA',
 'Low_NVDA',
 'Open_NVDA',
 'Volume_NVDA',
 'Daily_Gap',
 'sentiment',
 'sentiment_score']

Select the mode 'informative' new article for multiple per date

In [0]:
model = SentenceTransformer("all-MiniLM-L6-v2")

df_structured = df.withColumn("row_struct", struct(
    'Date','headline','summary','Close_NVDA','High_NVDA','Low_NVDA','Open_NVDA','Volume_NVDA','Daily_Gap','sentiment','sentiment_score'
))

grouped = df_structured.groupBy("Date").agg(collect_list("row_struct").alias("rows"))
grouped_rows = grouped.collect()

result_rows = []

for group in grouped_rows:
    date = group["Date"]
    rows = group["rows"]

    summaries = [row["summary"] for row in rows]

    if len(summaries) == 1:
        result_rows.append(rows[0])
        continue

    embeddings = model.encode(summaries)
    centroid = np.mean(embeddings, axis=0)
    distances = np.linalg.norm(embeddings - centroid, axis=1)
    best_index = np.argmax(distances)

    result_rows.append(rows[best_index])

schema = StructType([
    StructField("Date", StringType(), True),
    StructField("headline", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("Close_NVDA", StringType(), True),
    StructField("High_NVDA", StringType(), True),
    StructField("Low_NVDA", StringType(), True),
    StructField("Open_NVDA", StringType(), True),
    StructField("Volume_NVDA", StringType(), True),
    StructField("Daily_Gap", StringType(), True),
    StructField("sentiment", StringType(), True),
    StructField("sentiment_score", StringType(), True)
])

spark = SparkSession.builder.getOrCreate()
df_informative = spark.createDataFrame(result_rows, schema=schema)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
display(df_informative)

Date,headline,summary,Close_NVDA,High_NVDA,Low_NVDA,Open_NVDA,Volume_NVDA,Daily_Gap,sentiment,sentiment_score
2024-07-11,AMD’s Earnings Are Around the Corner. The Stock Is Getting More Love.,"The chip maker has outperformed recently, at least in part because of optimistic notes from Wall Street analysts",127.3686065673828,136.11644269274476,127.01869434277064,135.7165473660534,374782700,0.06151011767308378,Positive,0.9999858
2024-06-20,AMD Shares Now Top Piper Pick Because of AI Server Chips,Shares of Advanced Micro Devices traded higher Thursday after a Pipe Sandler analyst issued a bullish note on the artificial intelligence business of the chip maker.,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.9999987
2025-02-27,"AMD: Unrelenting Carnage, But A Tactical Trade Emerges","Discover why Advanced Micro Devices, Inc.'s stock drop presents a buying opportunity with growth prospects, improved margins and growth. Click for our AMD update.",120.15,135.01,120.01,135.0,443175846.0,0.10999999999999996,Positive,0.9999999
2024-07-30,"Down Between 17% and 35% From Their 52-Week Highs, Should You Buy the Dip on Nvidia, Broadcom, and AMD?",There are plenty of different ways to invest in artificial intelligence (AI) and growing demand for computing power.,103.7044448852539,111.96240417731258,102.5147356535528,111.49251876218644,486833300,0.0698528830758994,Positive,0.99874663
2025-01-07,"Are These Beaten-Down Stocks Worth A Look? AMD, NKE","Though the market has enjoyed a great stretch over the last year or so, not all have joined the party, including popular names such as Advanced Micro Devices and Nike. Is it time for a turnaround?",140.12710571289062,153.1159160516902,139.99711279126652,153.0159191493047,351782200,0.08423184664752338,Negative,0.8692298
2024-08-01,"AMD Q2: A 20% Pullback, Reset Expectations, And A Strong Revenue Growth Outlook (Rating Upgrade)","AMD shows consistent margin expansion, with 3Q guidance indicating at least a 25.1% non-GAAP operating margin. See why I upgrade AMD stock from sell to buy.",109.18309783935548,120.13040514788088,106.78368749587624,117.50104810323892,523462300,0.07079043462297549,Positive,1
2024-09-11,"AMD Gains Ground in AI Chip Market, Oracle Exec Confirms Growing Demand: Report","Advanced Micro Devices, Inc (NASDAQ:AMD) client Oracle Corp (NYSE:ORCL) acknowledged the chip designer gaining traction in the data center chip market for artificial intelligence despite Nvidia Corp’s moat. Oracle cloud business executive Karan Batta told the Information regarding customers’ openness to multiple vendors for inference computing implying opportunities for AMD and rivals. Recent reports indicated AMD is currently ditching the premium gaming GPU market for the mainstream and mid-ran",116.88119506835938,117.16112485124164,107.39352807236727,109.3630438515934,441422400,0.06874489728877854,Positive,0.9999838
2025-03-03,AMD: Consider It For Its Robust Growth Potential,Advanced Micro Devices is outperforming rivals with AI and data center growth. Learn why AMD stock is a smart buy near its 52-week lows.,114.06,123.7,112.28,123.51,411381373.0,0.07651202331794998,Positive,0.99999976
2024-08-05,Where Will AMD Stock Be in 1 Year?,The underdog chipmaker still has a bright future.,100.4252471923828,103.38452459369913,90.66765743904186,92.0373150047539,552842400,0.09113621129859817,Positive,0.9999988
2024-09-03,Jim Cramer Says Advanced Micro Devices Inc. (AMD) Is ‘Actually Very Cheap’,"We recently compiled a list of the Jim Cramer’s Top 10 Must-Watch Stocks Today. In this article, we are going to take a look at where Advanced Micro Devices Inc. (NASDAQ:AMD) stands against the other must-watch stocks according to Jim Cramer. Jim Cramer recently discussed Nvidia’s latest earnings report on Mad Money. Despite a solid quarter, […]",107.973388671875,116.18136480283336,107.26356453201906,115.9814171340765,477155100,0.06904578905898413,Negative,0.5434052


In [0]:
df_informative.groupBy("sentiment").count().show()

+---------+-----+
|sentiment|count|
+---------+-----+
| Positive|    9|
| Negative|    2|
+---------+-----+



In [0]:
prices = spark.sql("SELECT * FROM nvda_prices")
tags = df_informative

In [0]:
tags = tags.drop('headline','Close_NVDA','High_NVDA','Low_NVDA','Open_NVDA','Volume_NVDA')
tags.columns

Out[15]: ['Date', 'summary', 'Daily_Gap', 'sentiment', 'sentiment_score']

Helper method to help with word wrapping the summaries from the tags to condense the tooltips in the graph

In [0]:
def wrap_text(text, width=60):
    import textwrap
    return "<br>".join(textwrap.wrap(text, width=width)) if text else "None"


### Tagging Plot - AMD Tags on NVDA Prices

In [0]:
import plotly.graph_objects as go
import pandas as pd

# Assuming you're using Spark: convert to Pandas first
prices_pd = prices.toPandas()
tags_pd = tags.toPandas()

# Ensure 'Date' is in datetime format
prices_pd['Date'] = pd.to_datetime(prices_pd['Date'])
tags_pd['Date'] = pd.to_datetime(tags_pd['Date'])

# Merge tags into prices on 'Date'
merged = prices_pd.merge(tags_pd, on='Date', how='left')

# Convert columns to numeric
merged['close'] = pd.to_numeric(merged['close'], errors='coerce')
merged['open'] = pd.to_numeric(merged['open'], errors='coerce')
merged['volume'] = pd.to_numeric(merged['volume'], errors='coerce')
merged['sentiment_score'] = pd.to_numeric(merged['sentiment_score'], errors='coerce')

# Drop rows with missing close prices
merged = merged.dropna(subset=['close'])

# Tag indicator
merged['has_tag'] = merged['summary'].notnull()

# Remove sentiment data from untagged
merged.loc[~merged['has_tag'], ['summary', 'sentiment', 'sentiment_score']] = None

# Separate tagged and untagged
tagged = merged[merged['has_tag']]
# untagged = merged[~merged['has_tag']]

# Create custom hover text
def build_hover(df):
    return (
        "Date: " + df['Date'].astype(str) +
        "<br>Open: " + df['open'].astype(str) +
        "<br>Close: " + df['close'].astype(str) +
        "<br>Volume: " + df['volume'].astype(str) +
        "<br>Tag Summary: " + df['summary'].fillna("").apply(wrap_text) +
        "<br>Daily Gap: " + df['Daily_Gap'].astype(str).fillna("None") +
        "<br>Sentiment: " + df['sentiment'].fillna("None") +
        "<br>Sentiment Score: " + df['sentiment_score'].astype(str).fillna("None")
    )

# Create figure
fig = go.Figure()

# Untagged: lines + markers
fig.add_trace(go.Scatter(
    x=merged['Date'],
    y=merged['close'],
    mode='lines',
    name='Not Tagged',
    text=build_hover(merged),
    hoverinfo='text',
    line=dict(color='gray')
))

# Tagged: markers only
fig.add_trace(go.Scatter(
    x=tagged['Date'],
    y=tagged['close'],
    mode='markers',
    name='Tagged',
    text=build_hover(tagged),
    hoverinfo='text',
    marker=dict(size=10, color='red', symbol='diamond')
))

fig.update_layout(
    title="NVDA Stock Price with AMD Tag Indicator",
    xaxis_title="Date",
    yaxis_title="Close Price",
    legend_title="Tag Presence"
)

fig.show()
